In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle   #pickle a file to reuse it at deployment

In [2]:
## load the dataset

data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [ ]:
## preprocess the data
## drop irrelevant features
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data

In [6]:
## Geography and Gender are categorical variables
## encoding categorical variable (eg.Gender F-0 M-1)

label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


In [7]:
''' Using One hot encoding Geography column  (not using LabelEncoder as it will be 
 considered Spain (1) is greater than Germany(2) which is not accurate)
'''
from sklearn.preprocessing import OneHotEncoder
oneHotEncoder_geo=OneHotEncoder()
geo_encoder=oneHotEncoder_geo.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [8]:
oneHotEncoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [9]:
## sparse matrix in the form of dataframe  (geo_encoder.toarray() because originally it is sparse matrix)
geo_encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=oneHotEncoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [12]:
## combine one hot encoded data with original data
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [13]:
## save the encoder and onehot encoder pickle file
with open('label_encoder_gender','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('oneHotEncoder_geo','wb') as file:
    pickle.dump(oneHotEncoder_geo,file)

In [14]:
## divide the dataset into dependant and independant feature
## exited is a dependant feature rest are independant
y=data['Exited']    # dependant 
x=data.drop('Exited',axis=1)   # independant


## Split the data in training and testing sets
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42)

## scale these features
scaler=StandardScaler()
x_train=scaler.fit_transform(x_train)
x_test=scaler.fit_transform(x_test)

In [15]:
x_train

array([[ 0.21835119,  0.91186722,  1.91661905, ...,  1.00053348,
        -0.57776083, -0.57735027],
       [ 2.05728037,  0.91186722,  0.20210899, ..., -0.99946681,
         1.73082   , -0.57735027],
       [ 0.75860157, -1.09665089, -0.75039661, ...,  1.00053348,
        -0.57776083, -0.57735027],
       ...,
       [ 0.86249588, -1.09665089, -0.08364269, ...,  1.00053348,
        -0.57776083, -0.57735027],
       [ 0.15601461,  0.91186722,  0.3926101 , ...,  1.00053348,
        -0.57776083, -0.57735027],
       [ 0.46769752,  0.91186722,  1.15461458, ..., -0.99946681,
         1.73082   , -0.57735027]])

In [16]:
with open('scaler','wb') as file:
    pickle.dump(scaler,file)

In [17]:
y

0       1
1       0
2       1
3       0
4       0
       ..
9995    0
9996    0
9997    1
9998    1
9999    0
Name: Exited, Length: 10000, dtype: int64

In [39]:
data

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,1,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0
9996,516,1,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0
9997,709,0,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0
9998,772,1,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0


ANN Implementation

In [18]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [19]:
(x_train.shape[1],)  # no. of columns== no of input nodes
## above shows 12 inputs and single dimension

(12,)

In [20]:
## build our ANN model
model=Sequential([  ## one layer connected to other
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)),  ##HL1 connected with input layer
    Dense(32,activation='relu'), ##HL2
    Dense(1,activation='sigmoid')  ##output layer 
])

In [21]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [62]:
model.input_shape[1]

12

In [22]:
##  way of adding optimizers and loss

opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.BinaryCrossentropy()

In [23]:
## In order to do backwar and forwrd prapogation, we need to compile the model
model.compile(optimizer=opt,loss=loss,metrics=['accuracy']) 

In [36]:
## set up logs
log_dr="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")   ##strftime-->timme format

In [25]:
## setup tensorboard
from tensorflow.keras.callbacks import TensorBoard
tensorflow_callback=TensorBoard(log_dir=log_dr,histogram_freq=1)   ##histogram_freq= how often model logs the distribution of weights and biases

In [26]:
## setup early stopping (if loss value is not decreasing after certain epochs we can stop thetraining further)

early_stopping_callback=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)

In [27]:
## train the model
history= model.fit(
    x_train,y_train,validation_data=(x_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100


235/235 [==============================] - 3s 5ms/step - loss: 0.4059 - accuracy: 0.8328 - val_loss: 0.3575 - val_accuracy: 0.8512
Epoch 2/100
235/235 [==============================] - 1s 4ms/step - loss: 0.3573 - accuracy: 0.8543 - val_loss: 0.3617 - val_accuracy: 0.8540
Epoch 3/100
235/235 [==============================] - 1s 3ms/step - loss: 0.3495 - accuracy: 0.8573 - val_loss: 0.3466 - val_accuracy: 0.8600
Epoch 4/100
235/235 [==============================] - 1s 3ms/step - loss: 0.3439 - accuracy: 0.8577 - val_loss: 0.3511 - val_accuracy: 0.8624
Epoch 5/100
235/235 [==============================] - 1s 3ms/step - loss: 0.3387 - accuracy: 0.8595 - val_loss: 0.3448 - val_accuracy: 0.8600
Epoch 6/100
235/235 [==============================] - 1s 3ms/step - loss: 0.3367 - accuracy: 0.8596 - val_loss: 0.3543 - val_accuracy: 0.8632
Epoch 7/100
235/235 [==============================] - 1s 4ms/step - loss: 0.3342 - accuracy: 0.8624 - val_loss: 0.3393 - val_accuracy: 0.86

In [28]:
model.save('model.h5') ## h5 is compatible with keras

c:\Users\shrey\Anaconda3\envs\tfenv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [34]:
## to visualize tensorboard 
## load tensorboard extension

%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [ ]:
%tensorboard --logdir logs/fit

In [2]:
import sys
print(sys.executable)

c:\Users\shrey\Anaconda3\envs\tfenv\python.exe


In [ ]:
## load all the pickle files

